# Matched DID event study: four focal categories, excluding yogurt and milk

Save this notebook in `analysis/event_study/` and run cells in order.
The Windows paths below are already configured for this project.
The researcher must run it locally: the original parquet is not available here.

**Outcome:** healthy spending in four focal categories / total trip `food_spend`,
aggregated separately for each household-year. The four focal categories are
fresh fruits and vegetables, low-sugar cereal, healthy snacks, and healthy beverages.
Yogurt and milk are excluded from the healthy-spending numerator. The denominator
remains total annual `food_spend` across all food categories, including yogurt and
milk spending when present in the source data.
Outputs and private matched IDs use a separate `four_categories_no_dairy` subfolder.
No HMM posterior probabilities or outcome-code weights are used.

**Design:** separate analyses for six events; first *observed* occurrence,
no repeat of that focal event within [-3,+3], and a complete seven-year outcome
window. Controls have no observed occurrence of that focal event anywhere in
the input panel and a complete window in the same calendar years. This is
stricter than requiring no event only inside the window. Other life events
may occur in either group; analyses are not isolated single-event experiments.
With 2008–2017 data, full-window event cohorts are 2011–2014.

Match within each calendar cohort: 1:1 nearest propensity-score logit,
with replacement, exact pre-event region, and a 0.2-SD logit-distance caliper.
The propensity model uses Y at -3, -2, -1 plus age, income, household size,
and children at -1. No post-event covariates or outcomes enter matching scores.
Full post-event observation is an explicit sample restriction and can select
households based on attrition. Each retained treated household has equal weight.

At each k, estimate mean[(YT_k - YT_-1) - (YC_k - YC_-1)].
Cohorts are averaged by their matched treated counts, holding the same matched
sample at every k. Reference -1 is exactly zero. Pointwise 95% intervals use
household-clustered, fixed-design sandwich SEs, accounting for controls reused
across pairs/cohorts. These condition on the fitted matches; they do not include
propensity-score estimation or matching-selection uncertainty. A pre-period
joint Wald test is reported, but matching on pre-Y partly enforces pre-balance.
Causal interpretation additionally requires conditional parallel trends and
no anticipation; these plots alone establish adjusted associations.

Only aggregate tables/figures go to the public results folder. Annual panels
and matched household IDs remain in the separate, private data directory.
Software reference: [scikit-learn LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).


In [ ]:
# Cell 1 is deliberately standard-library-only: it does not import scientific
# libraries, scan parquet data, build font caches, or install packages.
print("[START] Python kernel is executing this cell.", flush=True)
from pathlib import Path
import os
import sys
import time
import json
import warnings
import importlib
import importlib.util
import importlib.metadata
import threading
from contextlib import contextmanager

# Set before importing NumPy/SciPy; restart the kernel for these to take effect.
for variable in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
                 "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ[variable] = "1"
print(f"Python: {sys.version.split()[0]} | Executable: {sys.executable}", flush=True)

@contextmanager
def progress(label):
    """Show elapsed time, not a fabricated completion percentage."""
    started = time.monotonic()
    done = threading.Event()
    print(f"[START] {label}", flush=True)
    def heartbeat():
        while not done.wait(15):
            print(f"[RUNNING] {label}: {time.monotonic()-started:.0f}s elapsed", flush=True)
    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    try:
        yield
    except BaseException:
        print(f"[STOPPED] {label}; see traceback below.", flush=True)
        raise
    else:
        print(f"[DONE] {label}: {time.monotonic()-started:.1f}s", flush=True)
    finally:
        done.set()
        worker.join(timeout=0.2)

def load_module(name):
    if name in sys.modules:
        return sys.modules[name]
    with progress(f"Import {name}"):
        return importlib.import_module(name)

DATA_PATH = Path(r"C:\Users\rwen13\Documents\healthy_purchase_data\processed\hmm_input\hmm_estimation_panel.parquet")
OUTPUT_DIR = Path(r"C:\Users\rwen13\Documents\healthy_purchase_cycle\model_free_evidence\four_categories_no_dairy")
PRIVATE_DIR = Path(r"C:\Users\rwen13\Documents\healthy_purchase_data\processed\model_free_evidence\four_categories_no_dairy")

YEAR_START, YEAR_END = 2008, 2017
PRE, POST, REFERENCE = 3, 3, -1
CALIPER_SD = 0.20
EXACT_REGION = True
MIN_TREATED, MIN_CONTROLS = 20, 20  # Eligibility thresholds before matching.
MIN_MATCHED = 20                   # Minimum retained pairs per cohort.
BALANCE_THRESHOLD = 0.10          # Diagnostic flag, not automatic deletion.
CHECK_DUPLICATE_TRIPS = True
USE_ANNUAL_CACHE = True
SAVE_PRIVATE_MATCHES = True
DUCKDB_MEMORY_LIMIT = "2GB"
DUCKDB_THREADS = min(4, max(1, (os.cpu_count() or 2) // 2))
AGGREGATION_MODE = "auto"  # auto, single_pass, or yearwise
RANDOM_SEED = 20261031
PIPELINE_VERSION = "1.2-four-categories-no-dairy"

EVENT_LABELS = {
    "event_new_child": "New child",
    "getting_married": "Marriage",
    "getting_divorce_or_separation": "Divorce / separation",
    "moving": "Moving",
    "getting_job": "Starting a job",
    "retirement": "Retirement",
}
HEALTHY_COLUMNS = {
    "fv_y": "fresh_spending",
    "cereal_y": "low_sugar_cereal_spending",
    "snack_y": "healthy_snack_spending",
    "beverage_y": "healthy_beverage_spending",
}
COVARIATES = ["age", "income10k_imputed", "family_size_numeric", "K_1"]
ANNUAL_COVARIATES = COVARIATES + (["regional_cd"] if EXACT_REGION else [])
EVENT_TIMES = list(range(-PRE, POST + 1))
assert PRE >= 2 and REFERENCE == -1 and POST >= 0

print("[READY] Settings loaded. No purchase data have been read.", flush=True)


## Startup instructions (repaired version)
**Restart the kernel first**, then run cells in order. Cell 1 prints START
immediately and uses only Python's standard library. If that line never
appears, the selected Jupyter kernel has not started executing Python; restart
or reselect the Python kernel in VS Code before running further cells.
Cell 2 lists missing dependencies and logs each import. It never installs or
upgrades packages automatically. If an import stalls, the last START message
identifies the module. First Matplotlib import is deferred until plotting,
with a writable local temporary font-cache directory.
Each scan/aggregation stage prints START/DONE plus an elapsed
time message every 15 seconds. Elapsed messages are activity indicators,
not proof that a slow native operation is making progress.


In [ ]:
packages = {"numpy": "numpy", "pandas": "pandas", "duckdb": "duckdb",
            "pyarrow": "pyarrow", "scipy": "scipy", "sklearn": "scikit-learn",
            "matplotlib": "matplotlib"}
missing = [pip_name for module, pip_name in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print("Run in a separate notebook cell, then restart the kernel:", flush=True)
    print("%pip install " + " ".join(missing), flush=True)
    raise ImportError("Missing packages: " + ", ".join(missing))
np = load_module("numpy")
pd = load_module("pandas")
duckdb = load_module("duckdb")
load_module("pyarrow")
from IPython.display import display
EVENT_TIMES = np.array(EVENT_TIMES, dtype=int)
print("[READY] Data libraries imported. Next cells define functions only.", flush=True)


## 1. Audit and aggregate trips with DuckDB
Missing healthy spending is set to zero only when that category's outcome is
missing (category nonpurchase). Missing spending on observed-category trips,
missing food spending, nonfinite/negative amounts, invalid events, varying
annual demographics/events, and duplicate trip keys stop execution.
Zero annual food spending produces missing Y and is excluded from matching.
Shares are not clipped, winsorized, smoothed, or rescaled for appearance.
The initial-2007 variable is not used in the outcome. Yogurt and milk are
excluded from the healthy-spending numerator. Total `food_spend` remains the
denominator and is not restricted to the four focal categories.
A cache keyed to source metadata and settings avoids repeated trip scans.


In [ ]:
def qi(name):
    return '"' + name.replace('"', '""') + '"'


def build_annual_panel(data_path, private_dir, table_dir):
    data_path, private_dir, table_dir = map(Path, (data_path, private_dir, table_dir))
    if not data_path.is_file():
        raise FileNotFoundError(f"Input parquet not found: {data_path}")
    private_dir.mkdir(parents=True, exist_ok=True)
    table_dir.mkdir(parents=True, exist_ok=True)
    stamp = {
        "pipeline_version": PIPELINE_VERSION,
        "source": str(data_path.resolve()),
        "size": data_path.stat().st_size,
        "mtime_ns": data_path.stat().st_mtime_ns,
        "healthy_columns": HEALTHY_COLUMNS,
        "annual_covariates": ANNUAL_COVARIATES,
        "years": [YEAR_START, YEAR_END],
        "duplicate_check": CHECK_DUPLICATE_TRIPS,
    }
    cache = private_dir / "annual_four_categories_no_dairy.parquet"
    meta = cache.with_suffix(".json")
    audit_cache = cache.with_suffix(".audit.csv")
    if USE_ANNUAL_CACHE and cache.exists() and meta.exists() and audit_cache.exists():
        if json.loads(meta.read_text(encoding="utf-8")) == stamp:
            panel = pd.read_parquet(cache)
            audit = pd.read_csv(audit_cache)
            audit.to_csv(table_dir / "input_audit.csv", index=False)
            print(f"Loaded validated annual cache: {len(panel):,} household-years")
            return panel, audit

    con = duckdb.connect()
    try:
        con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
        con.execute(f"SET threads={int(DUCKDB_THREADS)}")
        temp_sql = (private_dir / "duckdb_temp").as_posix().replace("'", "''")
        con.execute(f"SET temp_directory='{temp_sql}'")
        source_sql = data_path.as_posix().replace("'", "''")
        con.execute(f"CREATE VIEW trips AS SELECT * FROM read_parquet('{source_sql}')")
        names = set(con.execute("DESCRIBE trips").fetchdf()["column_name"])
        required = ({"hhid", "panel_year", "trip_id", "purchase_date", "food_spend"}
                    | set(EVENT_LABELS) | set(ANNUAL_COVARIATES)
                    | set(HEALTHY_COLUMNS) | set(HEALTHY_COLUMNS.values()))
        if required - names:
            raise ValueError(f"Missing required columns: {sorted(required - names)}")

        checks = {
            "trip_rows": "COUNT(*)",
            "missing_key_rows": "COUNT(*) FILTER (WHERE hhid IS NULL OR panel_year IS NULL OR trip_id IS NULL OR purchase_date IS NULL)",
            "outside_year_rows": f"COUNT(*) FILTER (WHERE panel_year NOT BETWEEN {YEAR_START} AND {YEAR_END})",
            "date_year_mismatch_rows": "COUNT(*) FILTER (WHERE YEAR(purchase_date) <> panel_year)",
            "invalid_food_rows": "COUNT(*) FILTER (WHERE food_spend IS NULL OR NOT isfinite(food_spend) OR food_spend < 0)",
            "zero_food_trip_rows": "COUNT(*) FILTER (WHERE food_spend = 0)",
        }
        for outcome, spend in HEALTHY_COLUMNS.items():
            checks[f"invalid__{spend}"] = f"COUNT(*) FILTER (WHERE ({qi(spend)} IS NULL AND {qi(outcome)} IS NOT NULL) OR ({qi(spend)} IS NOT NULL AND (NOT isfinite({qi(spend)}) OR {qi(spend)} < 0)))"
            checks[f"nonpurchase_null__{spend}"] = f"COUNT(*) FILTER (WHERE {qi(spend)} IS NULL AND {qi(outcome)} IS NULL)"
        for event in EVENT_LABELS:
            checks[f"invalid__{event}"] = f"COUNT(*) FILTER (WHERE {qi(event)} IS NULL OR {qi(event)} NOT IN (0,1))"
        expressions = [f"{expr} AS {qi(name)}" for name, expr in checks.items()]
        with progress("Audit trip amounts, keys and event codes"):
            raw = con.execute("SELECT " + ", ".join(expressions) + " FROM trips").fetchdf().iloc[0]
        audit_rows = [{"metric": k, "value": int(v)} for k, v in raw.items()]
        fatal = [k for k in raw.index if (k.startswith("invalid") or k in
                 ["missing_key_rows", "outside_year_rows", "date_year_mismatch_rows"]) and raw[k] > 0]
        if raw["trip_rows"] == 0:
            fatal.append("empty_input")
        if fatal:
            pd.DataFrame(audit_rows).to_csv(table_dir / "input_audit.csv", index=False)
            raise ValueError(f"Input audit failed; see input_audit.csv: {fatal}")

        healthy_sum = " + ".join(f"COALESCE({qi(c)},0)" for c in HEALTHY_COLUMNS.values())
        aggregates = ["COUNT(*) AS n_trips", "SUM(food_spend) AS food_spend",
                      f"SUM({healthy_sum}) AS healthy_spend"]
        if CHECK_DUPLICATE_TRIPS:
            aggregates.append("COUNT(DISTINCT (trip_id, purchase_date)) AS n_unique_trips")
        for c in list(EVENT_LABELS) + ANNUAL_COVARIATES:
            aggregates += [f"MIN({qi(c)}) AS {qi(c)}",
                           f"MAX({qi(c)}) AS {qi(c + '__max')}",
                           f"COUNT({qi(c)}) AS {qi(c + '__count')}"]
        # Inspect parquet metadata only. Broad year ranges in row groups imply
        # that filtering each year would repeatedly read much of the same file.
        mode = AGGREGATION_MODE
        if mode not in {"auto", "single_pass", "yearwise"}:
            raise ValueError("AGGREGATION_MODE must be auto, single_pass, or yearwise")
        if mode == "auto":
            pq = load_module("pyarrow.parquet")
            with pq.ParquetFile(data_path) as pf:
                col_idx = pf.schema_arrow.get_field_index("panel_year")
                spans = []
                for r in range(pf.metadata.num_row_groups):
                    stat = pf.metadata.row_group(r).column(col_idx).statistics
                    if stat is not None and stat.has_min_max:
                        spans.append(int(stat.max) - int(stat.min) + 1)
                mode = "yearwise" if spans and np.mean(spans) <= 2 else "single_pass"
        print(f"Aggregation mode: {mode}; DuckDB threads: {DUCKDB_THREADS}; memory cap: {DUCKDB_MEMORY_LIMIT}", flush=True)
        if mode == "single_pass":
            try:
                with progress("Aggregate all years in one scan, including duplicate-trip check"):
                    panel = con.execute("SELECT hhid, panel_year, " + ", ".join(aggregates)
                                        + " FROM trips GROUP BY hhid, panel_year").fetchdf()
            except duckdb.OutOfMemoryException:
                print("Memory cap reached; retrying year by year with the same checks.", flush=True)
                mode = "yearwise"
        if mode == "yearwise":
            yearly = []
            for year in range(YEAR_START, YEAR_END + 1):
                with progress(f"Aggregate year {year} (including duplicate-trip check)"):
                    part = con.execute("SELECT hhid, panel_year, " + ", ".join(aggregates)
                                       + f" FROM trips WHERE panel_year = {year}"
                                       + " GROUP BY hhid, panel_year").fetchdf()
                yearly.append(part)
                print(f"  {year}: {len(part):,} household-years", flush=True)
            panel = pd.concat(yearly, ignore_index=True)
    finally:
        con.close()

    fatal = []
    if CHECK_DUPLICATE_TRIPS:
        duplicates = int((panel.n_trips - panel.n_unique_trips).sum())
        audit_rows.append({"metric": "duplicate_trip_rows", "value": duplicates})
        if duplicates:
            fatal.append("duplicate_trip_rows")
        panel = panel.drop(columns="n_unique_trips")
    for c in list(EVENT_LABELS) + ANNUAL_COVARIATES:
        varying = int((panel[c].notna() & panel[c + "__max"].notna()
                       & (panel[c] != panel[c + "__max"])).sum())
        partial = int(((panel[c + "__count"] > 0)
                       & (panel[c + "__count"] < panel.n_trips)).sum())
        audit_rows += [{"metric": f"varying_within_year__{c}", "value": varying},
                       {"metric": f"partial_missing_within_year__{c}", "value": partial}]
        if varying or partial:
            fatal.append(c)
        panel = panel.drop(columns=[c + "__max", c + "__count"])
    panel["Y"] = panel.healthy_spend.div(panel.food_spend.where(panel.food_spend > 0))
    invalid_share = int(((panel.Y < -1e-10) | (panel.Y > 1 + 1e-10)).sum())
    audit_rows += [
        {"metric": "household_years", "value": len(panel)},
        {"metric": "households", "value": panel.hhid.nunique()},
        {"metric": "nonpositive_annual_food", "value": int((panel.food_spend <= 0).sum())},
        {"metric": "annual_share_outside_0_1", "value": invalid_share},
    ]
    if invalid_share:
        fatal.append("annual_share_outside_0_1")
    audit = pd.DataFrame(audit_rows)
    audit.to_csv(table_dir / "input_audit.csv", index=False)
    if fatal:
        raise ValueError(f"Annual audit failed; see input_audit.csv: {fatal}")
    panel = panel.sort_values(["hhid", "panel_year"]).reset_index(drop=True)
    panel.to_parquet(cache, index=False)
    audit.to_csv(audit_cache, index=False)
    meta.write_text(json.dumps(stamp, indent=2), encoding="utf-8")
    print(f"Aggregated {int(raw['trip_rows']):,} trips into {len(panel):,} household-years")
    return panel, audit


## 2. Match before estimating outcome changes
Exact region uses the **year -1** value, so a move in year 0 is not matched away.
A separate propensity model is fitted inside each eligible calendar cohort.
Scores are lightly regularized logistic regressions (C=1000), without class
reweighting. Common-support trimming uses the overlap of group score ranges.
The caliper is 0.2 times the pooled within-group SD of the original eligible
logit scores, held fixed before trimming. All matching rules are configurable.
Missing baseline features are excluded, not filled with post-event information.
Ties resolve deterministically by sorted control household ID.

The balance table reports cohort-specific standardized mean differences (SMDs)
before and after matching, using the SAME pre-match pooled SD denominator.
Controls repeated in matches receive their reuse frequency as weight.
Balance flags do not trigger hidden rematching or outcome-based model selection.


In [ ]:
def nearest_logit_matches(treated_scores, control_scores):
    """O(n log n) nearest neighbor search; controls must already be ID-sorted."""
    order = np.argsort(control_scores, kind="stable")
    sorted_scores = control_scores[order]
    # Keep the first ID among exact-score ties.
    unique_scores, starts = np.unique(sorted_scores, return_index=True)
    representatives = order[starts]
    right = np.searchsorted(unique_scores, treated_scores)
    a = np.clip(right - 1, 0, len(unique_scores) - 1)
    b = np.clip(right, 0, len(unique_scores) - 1)
    da, db = np.abs(treated_scores - unique_scores[a]), np.abs(treated_scores - unique_scores[b])
    chosen = np.where(da <= db, a, b)
    return representatives[chosen], np.minimum(da, db)


def match_event(panel, event, ywide):
    # Cached Python imports: heavy ML libraries load only when matching starts.
    StandardScaler = load_module("sklearn.preprocessing").StandardScaler
    LogisticRegression = load_module("sklearn.linear_model").LogisticRegression
    ConvergenceWarning = load_module("sklearn.exceptions").ConvergenceWarning
    ever = panel.groupby("hhid")[event].max().reindex(ywide.index)
    first = panel.loc[panel[event].eq(1)].groupby("hhid").panel_year.min()
    first = first.reindex(ywide.index)
    ewide = panel.pivot(index="hhid", columns="panel_year", values=event).reindex(ywide.index)
    pairs, diagnostics, balances = [], [], []
    for cohort in range(YEAR_START + PRE, YEAR_END - POST + 1):
        print(f"  {event}: cohort {cohort}", flush=True)
        window_years = cohort + EVENT_TIMES
        ywindow = ywide.reindex(columns=window_years)
        complete = np.isfinite(ywindow.to_numpy()).all(axis=1)
        ew = ewide.reindex(columns=window_years)
        treated_rule = first.eq(cohort) & ew.sum(axis=1).eq(1)
        control_rule = ever.eq(0)
        baseline = panel.loc[panel.panel_year.eq(cohort - 1)].set_index("hhid")
        features = pd.DataFrame(index=ywide.index)
        for k in range(-PRE, 0):
            features[f"Y_pre{abs(k)}"] = ywide.reindex(columns=[cohort+k]).iloc[:, 0]
        for c in ANNUAL_COVARIATES:
            features[c] = baseline[c].reindex(features.index)
        finite_x = np.isfinite(features.to_numpy(dtype=float)).all(axis=1)
        eligible = complete & finite_x
        t_ids = features.index[eligible & treated_rule.to_numpy()]
        c_ids = features.index[eligible & control_rule.to_numpy()]
        diag = {
            "event": event, "cohort": cohort,
            "first_observed_events": int(first.eq(cohort).sum()),
            "treated_no_repeat_complete_window": int((treated_rule.to_numpy() & complete).sum()),
            "eligible_treated": len(t_ids), "eligible_controls": len(c_ids),
            "matched_pairs": 0, "retained_pairs": 0,
            "status": "insufficient_eligible_sample",
        }
        if len(t_ids) < MIN_TREATED or len(c_ids) < MIN_CONTROLS:
            diagnostics.append(diag)
            continue
        ids = t_ids.append(c_ids)
        fit_features = features.loc[ids].copy()
        # Region enters the score as indicators and is also matched exactly.
        if EXACT_REGION:
            fit_features = pd.get_dummies(fit_features, columns=["regional_cd"], dtype=float)
        x = StandardScaler().fit_transform(fit_features)
        treatment = np.r_[np.ones(len(t_ids)), np.zeros(len(c_ids))]
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            fit = LogisticRegression(C=1000.0, solver="lbfgs", max_iter=2000,
                                     random_state=RANDOM_SEED).fit(x, treatment)
        score = pd.Series(fit.decision_function(x), index=ids)
        ts, cs = score.loc[t_ids], score.loc[c_ids]
        pooled_sd = np.sqrt((ts.var(ddof=1) + cs.var(ddof=1)) / 2)
        caliper = CALIPER_SD * pooled_sd
        lower, upper = max(ts.min(), cs.min()), min(ts.max(), cs.max())
        support = score.between(lower, upper)
        diag.update({"logit_caliper": caliper,
                     "treated_in_common_support": int(support.loc[t_ids].sum()),
                     "controls_in_common_support": int(support.loc[c_ids].sum())})
        t_support = t_ids[support.loc[t_ids].to_numpy()]
        c_support = c_ids[support.loc[c_ids].to_numpy()]
        cohort_pairs = []
        regions = sorted(features.loc[t_support, "regional_cd"].unique()) if EXACT_REGION else [None]
        for region in regions:
            ti, ci = t_support, c_support
            if EXACT_REGION:
                ti = ti[features.loc[ti, "regional_cd"].eq(region).to_numpy()]
                ci = ci[features.loc[ci, "regional_cd"].eq(region).to_numpy()]
            ci = ci.sort_values()
            if len(ti) == 0 or len(ci) == 0:
                continue
            nearest, distance = nearest_logit_matches(score.loc[ti].to_numpy(), score.loc[ci].to_numpy())
            keep = distance <= caliper + 1e-12
            cohort_pairs.append(pd.DataFrame({
                "event": event, "cohort": cohort,
                "treated_hhid": ti[keep], "control_hhid": ci[nearest[keep]],
                "logit_distance": distance[keep],
            }))
        matched = pd.concat(cohort_pairs, ignore_index=True) if cohort_pairs else pd.DataFrame()
        diag["matched_pairs"] = len(matched)
        if len(matched) < MIN_MATCHED:
            diag["status"] = "insufficient_matches_after_caliper"
            diagnostics.append(diag)
            continue
        diag.update({"status": "retained", "retained_pairs": len(matched),
                     "unique_controls": matched.control_hhid.nunique(),
                     "match_rate": len(matched) / len(t_ids)})
        diagnostics.append(diag)
        pairs.append(matched)
        pre_t = fit_features.loc[t_ids].to_numpy(dtype=float)
        pre_c = fit_features.loc[c_ids].to_numpy(dtype=float)
        post_t = fit_features.loc[matched.treated_hhid].to_numpy(dtype=float)
        post_c = fit_features.loc[matched.control_hhid].to_numpy(dtype=float)
        sd = np.sqrt((pre_t.var(axis=0, ddof=1) + pre_c.var(axis=0, ddof=1)) / 2)
        for j, name in enumerate(fit_features.columns):
            before, after = pre_t[:, j].mean() - pre_c[:, j].mean(), post_t[:, j].mean() - post_c[:, j].mean()
            smd_before = before / sd[j] if sd[j] > 0 else (0.0 if abs(before) < 1e-12 else np.nan)
            smd_after = after / sd[j] if sd[j] > 0 else (0.0 if abs(after) < 1e-12 else np.nan)
            balances.append({"event": event, "cohort": cohort, "feature": name,
                             "smd_before": smd_before, "smd_after": smd_after,
                             "absolute_smd_after": abs(smd_after),
                             "balance_flag": not np.isfinite(smd_after) or abs(smd_after) > BALANCE_THRESHOLD})
    pair_table = pd.concat(pairs, ignore_index=True) if pairs else pd.DataFrame()
    return pair_table, pd.DataFrame(diagnostics), pd.DataFrame(balances)


## 3. Estimate matched contrasts and household-clustered covariance
Let n_g be retained pairs in cohort g, N=sum(n_g), and dY=Y_k-Y_-1.
The estimate is sum_g (n_g/N)(mean_T,g(dY)-mean_C,g(dY)), where repeated
control appearances count repeatedly in the weighted control mean.
For each original household, sum its signed, cohort-mean-centered dY residuals
across ALL its matched appearances, divide by N, and take the cross-product
of these household influence vectors. Multiply by G/(G-1), where G is the
number of unique original households. This is the fixed-design clustered
sandwich for the cohort-specific weighted mean contrasts; cohort weights
and matches are held fixed. The covariance includes cross-event-time terms.
Reference -1 has zero estimate/SE by normalization and no p-value.


In [ ]:
def estimate_event(pair_table, ywide, event):
    stats = load_module("scipy.stats")
    n_total = len(pair_table)
    households = pd.Index(sorted(set(pair_table.treated_hhid) | set(pair_table.control_hhid)))
    influence = np.zeros((len(households), len(EVENT_TIMES)))
    estimate = np.zeros(len(EVENT_TIMES))
    cohort_rows = []
    ref_index = int(np.flatnonzero(EVENT_TIMES == REFERENCE)[0])
    for cohort, pairs in pair_table.groupby("cohort", sort=True):
        years = cohort + EVENT_TIMES
        yt = ywide.loc[pairs.treated_hhid, years].to_numpy(dtype=float)
        yc = ywide.loc[pairs.control_hhid, years].to_numpy(dtype=float)
        dt = yt - yt[:, [ref_index]]
        dc = yc - yc[:, [ref_index]]
        mt, mc = dt.mean(axis=0), dc.mean(axis=0)
        contrast = mt - mc
        estimate += len(pairs) / n_total * contrast
        np.add.at(influence, households.get_indexer(pairs.treated_hhid), (dt - mt) / n_total)
        np.add.at(influence, households.get_indexer(pairs.control_hhid), -(dc - mc) / n_total)
        cohort_rows.extend({"event": event, "cohort": int(cohort), "event_time": int(k),
                            "did_pp": 100 * contrast[j], "matched_treated": len(pairs),
                            "cohort_weight": len(pairs) / n_total}
                           for j, k in enumerate(EVENT_TIMES))
    g = len(households)
    covariance = influence.T @ influence * g / (g - 1)
    se = np.sqrt(np.maximum(np.diag(covariance), 0))
    critical = stats.t.ppf(0.975, g - 1)
    pvalues = np.full(len(EVENT_TIMES), np.nan)
    nonzero = se > 0
    pvalues[nonzero] = 2 * stats.t.sf(np.abs(estimate[nonzero] / se[nonzero]), g - 1)
    rows = pd.DataFrame({
        "event": event, "event_time": EVENT_TIMES,
        "did_pp": 100 * estimate, "se_pp": 100 * se,
        "ci_low_pp": 100 * (estimate - critical * se),
        "ci_high_pp": 100 * (estimate + critical * se),
        "p_value": pvalues, "matched_treated": n_total,
        "unique_controls": pair_table.control_hhid.nunique(),
        "household_clusters": g, "retained_cohorts": pair_table.cohort.nunique(),
        "is_reference": EVENT_TIMES == REFERENCE,
    })
    pre_idx = np.flatnonzero(EVENT_TIMES < REFERENCE)
    vpre = covariance[np.ix_(pre_idx, pre_idx)]
    rank = int(np.linalg.matrix_rank(vpre))
    # Do not report a seemingly valid joint test from a singular covariance.
    statistic, ppre = np.nan, np.nan
    if rank == len(pre_idx):
        bpre = estimate[pre_idx]
        statistic = float(bpre @ np.linalg.solve(vpre, bpre))
        ppre = float(stats.chi2.sf(statistic, rank))
    pretest = {"event": event, "wald_chi2": statistic, "df": rank,
               "pretrend_p_value": ppre, "full_rank": rank == len(pre_idx)}
    cov_rows = [{"event": event, "event_time_1": int(k), "event_time_2": int(l),
                 "covariance_pp_squared": 10000 * covariance[j, m]}
                for j, k in enumerate(EVENT_TIMES) for m, l in enumerate(EVENT_TIMES)]
    return rows, pretest, pd.DataFrame(cohort_rows), pd.DataFrame(cov_rows)


## 4. Plot six events with a common, data-driven Y axis
No smoothing or selective CI removal. Axis limits include every interval,
are symmetric around zero, and use about five major ticks. If an event has
no eligible matches, its panel says so. The orange line is at year 0;
-1 is the omitted reference, drawn as a hollow point at zero.


In [ ]:
def plot_event_study(results, output_dir):
    import tempfile
    # Avoid a slow/network-mounted default Matplotlib configuration directory.
    mpl_cache = Path(tempfile.gettempdir()) / "healthy_purchase_matplotlib"
    mpl_cache.mkdir(parents=True, exist_ok=True)
    os.environ["MPLCONFIGDIR"] = str(mpl_cache)
    global plt
    plt = load_module("matplotlib.pyplot")
    ticker = load_module("matplotlib.ticker")
    MaxNLocator, FuncFormatter = ticker.MaxNLocator, ticker.FuncFormatter
    plt.rcParams.update({"font.family": "serif", "font.size": 10,
                         "axes.spines.top": False, "axes.spines.right": False,
                         "pdf.fonttype": 42, "ps.fonttype": 42})
    finite_ci = results[["ci_low_pp", "ci_high_pp"]].to_numpy().ravel() if len(results) else np.array([])
    finite_ci = finite_ci[np.isfinite(finite_ci)]
    bound = float(np.max(np.abs(finite_ci))) if finite_ci.size else 0.1
    bound = max(bound * 1.12, 0.01)
    locator = MaxNLocator(nbins=4, steps=[1, 2, 2.5, 5, 10], symmetric=True)
    ticks = locator.tick_values(-bound, bound)
    fig, axes = plt.subplots(2, 3, figsize=(12.6, 7.4), sharex=True, sharey=True)
    for ax, (event, label) in zip(axes.flat, EVENT_LABELS.items()):
        sub = results.loc[results.event.eq(event)].sort_values("event_time") if len(results) else pd.DataFrame()
        ax.axhline(0, color="#737373", linewidth=0.9, zorder=1)
        ax.axvline(0, color="#D58B34", linewidth=1.4, alpha=0.95, zorder=2)
        if len(sub):
            nonref = sub.loc[~sub.is_reference]
            ax.errorbar(nonref.event_time, nonref.did_pp,
                        yerr=np.vstack([nonref.did_pp - nonref.ci_low_pp,
                                        nonref.ci_high_pp - nonref.did_pp]),
                        fmt="o", color="#245A81", markersize=4.5,
                        elinewidth=1.15, capsize=3, zorder=4)
            ax.plot(sub.event_time, sub.did_pp, color="#245A81", linewidth=1.15, zorder=3)
            ax.scatter([REFERENCE], [0], facecolor="white", edgecolor="#245A81", s=29, zorder=5)
            n, c = int(sub.matched_treated.iloc[0]), int(sub.unique_controls.iloc[0])
            ax.text(0.03, 0.96, f"Treated: {n:,} | Controls: {c:,}",
                    transform=ax.transAxes, va="top", fontsize=8, color="#555555")
        else:
            ax.text(0.5, 0.5, "Insufficient eligible matches", ha="center", transform=ax.transAxes)
        ax.set_title(label, fontsize=11, pad=11)
        ax.set_xticks(EVENT_TIMES)
        ax.set_xlim(-PRE - 0.3, POST + 0.3)
        ax.set_yticks(ticks)
        ax.set_ylim(ticks[0], ticks[-1])
        ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:.4f}".rstrip("0").rstrip(".") if abs(x) > 1e-10 else "0"))
        ax.grid(axis="y", color="#E6E6E6", linewidth=0.6)
        ax.set_axisbelow(True)
    fig.supylabel("Change in healthy focal-category spending share (pp)", x=0.018, fontsize=11)
    fig.supxlabel("Years relative to event (reference: −1)", y=0.083, fontsize=11)
    fig.text(0.065, 0.023,
             "Matched DID; pointwise 95% household-clustered CIs conditional on matching. "
             "Orange line: event year.\n"
             "Equal weight per matched treated household; controls may be reused. "
             "Reference year −1 is normalized to zero.", fontsize=8, color="#555555")
    fig.subplots_adjust(left=0.085, right=0.985, top=0.94, bottom=0.18, wspace=0.12, hspace=0.32)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_dir / "matched_did_six_events.png", dpi=300, bbox_inches="tight")
    fig.savefig(output_dir / "matched_did_six_events.pdf", bbox_inches="tight")
    return fig


## 5. Run the analysis and save aggregate results
Public output: `tables/`, `figures/`, and a methods/configuration record.
Private output: validated annual parquet, cache metadata, and optional matched
household-ID files, all under the separate data directory.
Always inspect sample retention, control reuse, and balance before interpreting
an event plot. A missing event panel is retained rather than silently omitted.


In [ ]:
def run_analysis():
    table_dir, figure_dir = OUTPUT_DIR / "tables", OUTPUT_DIR / "figures"
    table_dir.mkdir(parents=True, exist_ok=True)
    figure_dir.mkdir(parents=True, exist_ok=True)
    annual, audit = build_annual_panel(DATA_PATH, PRIVATE_DIR, table_dir)
    display(audit)
    summary = annual.groupby("panel_year").agg(
        households=("hhid", "nunique"), trips=("n_trips", "sum"),
        valid_outcome_households=("Y", "count"),
        mean_annual_share=("Y", "mean"), median_annual_share=("Y", "median"),
    ).reset_index()
    summary.to_csv(table_dir / "annual_sample_summary.csv", index=False)
    display(summary)
    ywide = annual.pivot(index="hhid", columns="panel_year", values="Y").reindex(
        columns=range(YEAR_START, YEAR_END + 1))
    results, diagnostics, balances, pretests, cohort_results, covariances = [], [], [], [], [], []
    statuses = []
    for event, label in EVENT_LABELS.items():
        print(f"Matching: {label} ...", flush=True)
        pairs, diag, balance = match_event(annual, event, ywide)
        diagnostics.append(diag)
        if not balance.empty:
            balances.append(balance)
        if pairs.empty:
            statuses.append({"event": event, "status": "no_retained_cohort", "matched_treated": 0})
            print("  No cohort met the matching thresholds.")
            continue
        if SAVE_PRIVATE_MATCHES:
            match_dir = PRIVATE_DIR / "matched_household_ids"
            match_dir.mkdir(parents=True, exist_ok=True)
            pairs.to_parquet(match_dir / f"{event}.parquet", index=False)
        estimates, pretest, cohort_table, covariance = estimate_event(pairs, ywide, event)
        results.append(estimates)
        pretests.append(pretest)
        cohort_results.append(cohort_table)
        covariances.append(covariance)
        statuses.append({"event": event, "status": "estimated", "matched_treated": len(pairs),
                         "unique_controls": pairs.control_hhid.nunique(),
                         "max_control_reuse": int(pairs.control_hhid.value_counts().max()),
                         "control_effective_sample_size": len(pairs)**2 / float((pairs.control_hhid.value_counts()**2).sum())})
        print(f"  Retained {len(pairs):,} treated households; {pairs.control_hhid.nunique():,} unique controls.")
    result_table = pd.concat(results, ignore_index=True) if results else pd.DataFrame(columns=["event"])
    balance_table = pd.concat(balances, ignore_index=True) if balances else pd.DataFrame(
        columns=["event", "cohort", "feature", "smd_before", "smd_after", "absolute_smd_after", "balance_flag"])
    diagnostic_table = pd.concat(diagnostics, ignore_index=True)
    files = {
        "event_study_coefficients.csv": result_table,
        "matching_sample_flow.csv": diagnostic_table,
        "matching_balance.csv": balance_table,
        "event_status.csv": pd.DataFrame(statuses),
        "pretrend_joint_tests.csv": pd.DataFrame(pretests, columns=["event", "wald_chi2", "df", "pretrend_p_value", "full_rank"]),
        "cohort_event_study.csv": pd.concat(cohort_results, ignore_index=True) if cohort_results else pd.DataFrame(columns=["event"]),
        "event_study_covariance.csv": pd.concat(covariances, ignore_index=True) if covariances else pd.DataFrame(columns=["event"]),
    }
    for filename, frame in files.items():
        # Guard against accidental household-ID exports to the public folder.
        if any("hhid" in str(c).lower() for c in frame.columns):
            raise RuntimeError(f"Household identifier in aggregate export: {filename}")
        frame.to_csv(table_dir / filename, index=False)
    if not balance_table.empty:
        balance_summary = balance_table.groupby("event").agg(
            max_absolute_smd_after=("absolute_smd_after", "max"),
            flagged_feature_cohort_cells=("balance_flag", "sum"),
        ).reset_index()
        balance_summary.to_csv(table_dir / "matching_balance_summary.csv", index=False)
        display(balance_summary)
        if balance_table.balance_flag.any():
            warnings.warn("Some matched covariates have |SMD| > 0.10 or undefined balance; inspect matching_balance.csv.")
    config = {
        "pipeline_version": PIPELINE_VERSION, "input_filename": DATA_PATH.name,
        "focal_categories": ["fresh fruits and vegetables", "low-sugar cereal", "healthy snacks", "healthy beverages"],
        "excluded_focal_categories": ["yogurt", "milk"],
        "healthy_spending_columns": list(HEALTHY_COLUMNS.values()),
        "denominator": "sum of trip food_spend within household-year across all food categories",
        "year_start": YEAR_START, "year_end": YEAR_END,
        "event_window": [-PRE, POST], "reference": REFERENCE,
        "matching": "1:1 nearest logit propensity score, with replacement, within calendar cohort",
        "propensity_C": 1000, "baseline_features": [f"Y_pre{k}" for k in range(PRE, 0, -1)] + COVARIATES,
        "exact_region_at_minus_one": EXACT_REGION, "caliper_pooled_logit_sd": CALIPER_SD,
        "controls": "no focal event observed anywhere in input panel",
        "treated": "first observed focal event; no repeat within complete event window",
        "minimum_eligible_treated": MIN_TREATED, "minimum_eligible_controls": MIN_CONTROLS,
        "minimum_retained_pairs_per_cohort": MIN_MATCHED,
        "duplicate_trip_check": CHECK_DUPLICATE_TRIPS,
        "aggregation_mode": AGGREGATION_MODE, "duckdb_threads": DUCKDB_THREADS,
        "inference": "fixed-match cohort-mean sandwich clustered by original household; G/(G-1) correction; pointwise t CIs",
        "first_stage_uncertainty_included": False,
        "versions": {p: importlib.metadata.version(p) for p in
                     ["duckdb", "pandas", "numpy", "scipy", "scikit-learn", "matplotlib", "pyarrow"]},
    }
    (OUTPUT_DIR / "analysis_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    (OUTPUT_DIR / "README.md").write_text(
        "# Matched DID event study\n\n"
        "Outcome: annual healthy spending in four focal categories (fresh fruits and vegetables, "
        "low-sugar cereal, healthy snacks, and healthy beverages) / annual sum of trip food_spend. "
        "Yogurt and milk are excluded from the numerator; the denominator remains total food spending. "
        "See analysis_config.json for category inclusion and matching settings.\n\n"
        "The six event analyses use first observed events, no focal-event recurrence in the window, "
        "never-observed-event controls, complete outcome windows, and pre-event matching only. "
        "The analysis estimates dynamic matched changes relative to year -1, in percentage points. "
        "The orange vertical line marks year 0. These are adjusted associations; "
        "causal interpretation requires additional assumptions.\n\n"
        "Pointwise 95% CIs use household-clustered fixed-match sandwich SEs. "
        "Control reuse across cohorts is included; propensity estimation and matching uncertainty are not. "
        "Pretrend tests are partly constrained by matching on pre-event outcomes. "
        "Full-window restrictions select on observation/attrition.\n\n"
        "Inspect tables/matching_sample_flow.csv and tables/matching_balance.csv before interpreting figures. "
        "Counts are unweighted household counts; no Nielsen projection weights are used. "
        "Annual mean shares are unweighted household means, not aggregate dollar market shares.\n\n"
        "This folder contains aggregate outputs only. Annual microdata and matched household IDs "
        "are saved separately under the private data directory.\n",
        encoding="utf-8")
    fig = plot_event_study(result_table, figure_dir)
    display(pd.DataFrame(statuses))
    display(pd.DataFrame(pretests))
    plt.show()
    print(f"Finished. Aggregate outputs: {OUTPUT_DIR}")
    return {"annual_panel": annual, "coefficients": result_table,
            "sample_flow": diagnostic_table, "balance": balance_table, "figure": fig}


print("[START] Full analysis: audit -> annual aggregation -> matching -> figures", flush=True)
analysis = run_analysis()
